In [2]:
# 데이터 불러오기

from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
# 학습 데이터 불러오기
train_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=True, # train 인지 
            download=True,# 다운로드 해야 함
            transform=ToTensor() # 이렇게 바꿔줌
)

In [4]:
# 테스트 데이터 불러오기
test_data = datasets.FashionMNIST(
            root="../Data" , # 어디에 저장할 건지
            train=False,
            download=True,
            transform=ToTensor()
)

In [5]:
# 받은 데이터 확인
print(train_data.data.shape)
print(train_data.targets.shape)
print(test_data.data.shape)
print(test_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])
torch.Size([10000, 28, 28])
torch.Size([10000])


In [6]:
# Train과 data를 훈련 데이터와 검증데이터로 나누기

from sklearn.model_selection import train_test_split
# Cnn이라 채널을 추가해줘야 한다 (컬러채널)

train_input = train_data.data.unsqueeze(1).float() / 255.0 # 채널 차원 추가 및 정규화
train_target = train_data.targets # 이름이 길어서 이렇게 해줌

test_input = test_data.data.unsqueeze(1).float() / 255.0 # 채널 차원 추가 및 정규화
test_target = test_data.targets # 이름이 길어서 이렇게 해줌

train_input, val_input, train_target, val_target = train_test_split(
                                                        train_input,
                                                        train_target,
                                                        test_size=0.2,
                                                        random_state=42
)

In [7]:
# Dimension

print(train_input.shape, train_target.shape)
print(test_input.shape, test_target.shape)

torch.Size([48000, 1, 28, 28]) torch.Size([48000])
torch.Size([10000, 1, 28, 28]) torch.Size([10000])


---
#### CNN 신경망 만들기

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [9]:
# Dataset과 Dataloader 생성

batch_size = 32 # mini batch
train_dataset = TensorDataset(train_input, train_target)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True) # 한번 epoch가 발생했을때 섞어 쓰는 거

val_dataset = TensorDataset(val_input, val_target)
val_loader = DataLoader(val_dataset, batch_size=batch_size) # 벨리드나 테스트는 검증이라 셔플을 쓰지 않는다 

#### 모델 정의

In [10]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__() # super에 있는 모델을 쓰겠다는 거 
        # flatten 층 위에 많이 들어감

        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # flatten 층_과 밀집층
        self.flatten = nn.Flatten() # 층을 펴주는 거 
        self.fc1 = nn.Linear(64*7*7, 128)
        self.relu3 = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128,10) # 512개 들어와서 10개로 준다 
        self.softmax = nn.Softmax(dim=1) # 앞에는 변수임
    
    def forward(self, x) : 
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x) # 들어온 데이터로 층 만든것
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        x = self.softmax(x)
        return self.softmax(x)

In [11]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [12]:
# 모델, 손실함수, 옵티마이져

model = CNNModel().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

---
#### 모델 훈련

In [13]:
# 학습 함수
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device) # 디바이스 (cpu)로 보냄
        optimizer.zero_grad() # 기울기 초기화 시켜주는 거 _옵티마이저가 곱하기 하는거라서 
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    return loss.item()

In [ ]:
# # 학습 함수
# def train(model, train_loader, criterion, optimizer, device):
#     model.train()
#     for inputs, targets in train_loader:
#         inputs, targets = inputs.to(device), targets.to(device)
#         optimizer.zero_grad()
#         outputs = model(inputs)
#         loss = criterion(outputs, targets)
#         loss.backward()
#         optimizer.step()
#     return loss.item()

In [14]:
model.to(device)

CNNModel(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (relu3): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

In [15]:
# 훈련하기
num_epochs = 50

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, criterion, optimizer, device)
    print(f'Epoch[{epoch+1:>3} / {num_epochs}], Loss : {train_loss:.4f}')

Epoch[  1 / 50], Loss : 2.2541
Epoch[  2 / 50], Loss : 2.2264
Epoch[  3 / 50], Loss : 2.2448
Epoch[  4 / 50], Loss : 2.2388
Epoch[  5 / 50], Loss : 2.2311
Epoch[  6 / 50], Loss : 2.2449
Epoch[  7 / 50], Loss : 2.2273
Epoch[  8 / 50], Loss : 2.1955
Epoch[  9 / 50], Loss : 2.1846
Epoch[ 10 / 50], Loss : 2.2080
Epoch[ 11 / 50], Loss : 2.1991
Epoch[ 12 / 50], Loss : 2.1887
Epoch[ 13 / 50], Loss : 2.1717
Epoch[ 14 / 50], Loss : 2.1911
Epoch[ 15 / 50], Loss : 2.1893
Epoch[ 16 / 50], Loss : 2.1746
Epoch[ 17 / 50], Loss : 2.1854
Epoch[ 18 / 50], Loss : 2.2037
Epoch[ 19 / 50], Loss : 2.1829
Epoch[ 20 / 50], Loss : 2.1895
Epoch[ 21 / 50], Loss : 2.1808
Epoch[ 22 / 50], Loss : 2.1854
Epoch[ 23 / 50], Loss : 2.1821
Epoch[ 24 / 50], Loss : 2.1808
Epoch[ 25 / 50], Loss : 2.1852
Epoch[ 26 / 50], Loss : 2.1762
Epoch[ 27 / 50], Loss : 2.1808
Epoch[ 28 / 50], Loss : 2.1716
Epoch[ 29 / 50], Loss : 2.1808
Epoch[ 30 / 50], Loss : 2.1804
Epoch[ 31 / 50], Loss : 2.1716
Epoch[ 32 / 50], Loss : 2.1808
Epoch[ 3

In [17]:
# 평가함수 
def evaluate(model,val_loader,criterion,device):
    model.eval()
    total_loss = 0 # 전체 손실 합계 
    correct = 0  # 정확하게 예측한 샘플 수 
    total = 0 # 전체 샘플 수
    with torch.no_grad():
        for inputs ,targets in val_loader: # 문제 정답 넣기
            inputs,targets = inputs.to(device),targets.to(device) # 문제 정답 디바이스로 보내기
            outputs = model(inputs)
            loss = criterion(outputs,targets)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss / len(val_loader), correct / total

In [18]:
# 훈련 평가
train_loss, train_accuarcy = evaluate(model, train_loader, criterion, device)
print(f"Loss : {train_loss}, Accuracy : {train_accuarcy}")

Loss : 2.179351958433787, Accuracy : 0.9475


In [19]:
# 일반화 평가
test_dataset = TensorDataset(test_input, test_target)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


#검증 평가
val_loss, val_accuarcy = evaluate(model, val_loader, criterion, device)
print(f"Loss : {val_loss}, Accuracy : {val_accuarcy}")

Loss : 2.1843039010365803, Accuracy : 0.9136666666666666


---
#### 학습한 모델 저장하기

In [16]:
# 전체 모델 저장
torch.save(model, "../Data/cnn_fsmist.pth")

In [20]:
# 전체 모델 불러오기
model1 = torch.load("../Data/cnn_fsmist.pth", weights_only=False)

---
#### Image 만들어서 예측해보기

In [21]:
from PIL import Image
import numpy as np

In [22]:
# train data의 50번째
abc = np.array(train_data.data[50]).reshape(28,28)
abc.shape

/var/folders/7v/t26g0nsx69l66fs_rjzzq1s00000gn/T/ipykernel_61019/2095174428.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  abc = np.array(train_data.data[50]).reshape(28,28)


(28, 28)

In [23]:
# Image 만들기
image = Image.fromarray(abc)
image

In [24]:
# Image 저장
image.save("../Data/fashion_mnist_50.png")

---
#### Image를 불러서 예측

In [26]:
img = Image.open("../Data/fashion_mnist_50.png")
img

In [27]:
## image를 numpy array
img = np.array(img)
img = torch.from_numpy(img) # 이미지-> 넘파이->토치
img

tensor([[  0,   0,   0,   0,   0,   0,   0,   0,   1,   0,   0,   9,   6,   0,
           0,   0,  24,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   0,  67, 209, 231, 248, 252,
         250, 253, 246, 206, 132,   0,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,   1, 227, 243, 234, 234, 248,
         246, 238, 230, 234, 250, 126,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,  57, 231, 213, 227, 234, 232,
         231, 235, 232, 218, 218, 222,   0,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0,  90, 239, 207, 213, 236, 235,
         232, 232, 229, 210, 215, 207,   6,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   0, 211, 245, 229, 197, 220, 221,
         221, 222, 203, 221, 235, 222,  96,   0,   0,   0,   0,   0,   0,   0],
        [  0,   0,   0,   0,   0,   0,   0,   

In [28]:
# 채널 차원 추가 및 정규화
img = img.unsqueeze(0).float() / 255.0
img.shape

torch.Size([1, 28, 28])

In [29]:
# class들의 이름 정의
classes = ['티셔츠', '바지', '스웨터', '드레스', '코트', '샌달', '셔츠', '스니커즈', '가방', '앵글 부츠']

In [30]:
# 단일 데이터의 예측 함수
def predictOne(model, image, device, classes):
    model.eval()
    with torch.no_grad():
        image = image.to(device)
        outputs = model(image.unsqueeze(0)) # 차원 추가_ 학습을 3개 차원으로 해서 / 언스퀴즈 추가, 제거 스퀴즈 (1인 것만 뺄 수 있음)
        _, predicted = torch.max(outputs, 1)
        predicted_classes = classes[predicted.item()]
    return predicted_classes

In [31]:
predictOne(model, img, device, classes)

'드레스'